# cintx GTH-MOLOPT 2e profile on a Colab T4

`ci/colab_t4_verification.ipynb` settles whether the CUDA backend produces
libcint's numbers. This notebook is the **measurement** half for the batched
`int2e_sph` path over the GTH-MOLOPT bases (`DZVP-MOLOPT-SR-GTH`,
`TZVP-MOLOPT-GTH`), run the way the CubeCL profiling manual prescribes:

1. correctness first (`def2_cuda_verification`, and the CPU-pinned
   cooperative-arm and ket-split gates),
2. portable timing as **in-process A/B ratios** (`gth_profile`, every variant
   one compiled program),
3. attribution with vendor counters — `nsys` timeline, `ncu` memory-pipeline
   metrics — on the 2e kernel `two_electron_scalar_kernel_f_f64`,
4. everything written to one directory to paste back.

**What the A/B rows mean on a discrete GPU**

| variant | question |
|---|---|
| `klsplit=off` | one quartet per cube — the shape `gth_molopt_speed_memory_plan.md` §10.5 found latency-bound on the integrated gfx1151. The default spreads each quartet's ket-pair range over enough cubes to give every SM several workgroups (G1). The ratio is the split's worth on 40 SMs. |
| `probe:no-ctr` | the G build without the contraction; `1/ratio` is the contraction's share of the kernel after K1. |
| `coop=lane0` | the pre-S3 G build on one lane; the ratio is what splitting the build by `(axis, root)` buys. |
| `naive` | the pre-C1 contraction; its ratio says how much of the staged scheme's win survives on this device. |
| `xform=device` | M3: the cart→sph transform on the device, so the readback is the spherical output. On a discrete GPU the readback saving is a real bus transfer. |

**What the counters answer** (manual §4): whether the kernel is bound by
memory sectors (`bytes_per_sector`, hit rates), by f64 issue
(`dfma` instruction share — a T4 runs f64 at 1/32 of f32), or by barrier /
scoreboard stalls (`warp_issue_stalled_*`). That is what decides the next
kernel change: primitive-quartet parallelism inside the cube, shared-memory
G, or neither.

A T4's f64 rate is ~254 GFLOP/s. **Absolute times describe the T4**; the
ratios describe the kernel.


## 1. Confirm the GPU and the profilers

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv
!which nsys ncu || ls /usr/local/cuda/bin | grep -E 'nsys|ncu' || echo 'profilers not on PATH; the script searches /usr/local/cuda'

## 2. Get the source

cintx is not published, so bring your own copy: clone your remote or upload a
tarball and untar it to `/content/cintx`.

In [ ]:
# Option A — clone (replace with your remote)
# !git clone --depth 1 <your-cintx-remote> /content/cintx

# Option B — upload cintx.tar.gz via the Files pane, then:
# !mkdir -p /content/cintx && tar xzf /content/cintx.tar.gz -C /content/cintx --strip-components=1

import pathlib
assert pathlib.Path('/content/cintx/Cargo.toml').exists(), \
    'put the repo at /content/cintx first (clone or untar above)'
print('source present')

## 3. Install Rust

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version && rustc --version

## 4. Build, verify, time, attribute

One script, so the run is reproducible outside a notebook. It builds with
`--features cpu,cuda,extended-device-rys,gth` and `CINTX_ORACLE_BUILD_VENDOR=1`
(vendored libcint 6.1.3 as the oracle), runs the correctness gates, the
`gth_profile` A/B on CUDA and on the VM's CPU backend, then `nsys` and `ncu`
on the 2e kernel. `CINTX_T4_FIXTURES` filters the GTH fixtures by label
(`H2O` is the 30-minute budget; `CH4` and `SO2` are several times longer);
`CINTX_T4_SKIP_NCU=1` skips the counter replay.

In [ ]:
!CINTX_T4_FIXTURES=H2O bash /content/cintx/ci/colab_t4_profile.sh 2>&1 | tail -120

## 5. Read the summary

In [ ]:
import json, pathlib
out = pathlib.Path('/content/cintx_t4_profile')
summary = json.loads((out / 'summary.json').read_text())
print(json.dumps(summary['profile_cuda'], indent=2))
for kernel, metrics in summary['ncu'].items():
    print('\n', kernel)
    for name, value in metrics.items():
        print(f'  {name:<70} {value}')

## 6. Optional — the wider fixtures and the full `ncu` report

`CH4` (2 211 quartets) and `SO2` (1 035 quartets, sulfur's wider ket ranges)
are where the split has less to do — more cubes already — so their
`klsplit=off` ratios bound the split's worth from the other side. The
`--set full` report (`two_electron_full.ncu-rep`) opens in Nsight Compute for
the guided memory-workload analysis.

In [ ]:
# !CINTX_T4_FIXTURES=CH4 CINTX_T4_SKIP_NCU=1 bash /content/cintx/ci/colab_t4_profile.sh 2>&1 | tail -60

## 7. What to bring back

Paste `summary.json` and the tails of `step2_gth_profile_cuda.log` and
`step3_ncu.log`. The lines that decide the next kernel change:

- the `klsplit=off` ratio per fixture, and `kl_split=` on the launches line —
  the split's worth and how wide it went on 40 SMs;
- `probe:no-ctr` — the contraction's remaining share;
- `smsp__warp_issue_stalled_barrier…` against `…long_scoreboard…` — whether the
  two barriers per primitive quartet or the G-tensor loads are the stall;
- `sm__sass_average_data_bytes_per_sector_mem_global_op_ld.pct` — whether the
  G-tensor and coefficient loads are coalesced (the index table makes lane
  `q` read three scattered offsets; a low figure here is the case for a
  shared-memory G on NVIDIA, which §9.2 found a wash on AMD);
- `launch__registers_per_thread` and `launch__occupancy_limit_registers` — the
  private Rys/accumulator arrays' cost in occupancy.
